In [1]:
from google.colab import drive
drive.mount('/content/drive')

import numpy as np, pandas as pd, sklearn, os
print("numpy:", np.__version__, "pandas:", pd.__version__, "sklearn:", sklearn.__version__)

# Auto-locate the CSV anywhere under MyDrive
TARGET_FILENAME = "openfoodfacts.csv"
ROOT = "/content/drive/MyDrive"

found = []
for dirpath, _, filenames in os.walk(ROOT):
    if TARGET_FILENAME in filenames:
        found.append(os.path.join(dirpath, TARGET_FILENAME))
if not found:
    raise FileNotFoundError(f"Could not find '{TARGET_FILENAME}' under {ROOT}. Put the file in Drive and re-run.")
CSV_PATH = found[0]
print("Using:", CSV_PATH)


Mounted at /content/drive
numpy: 2.0.2 pandas: 2.2.2 sklearn: 1.6.1
Using: /content/drive/MyDrive/ml_assignment/openfoodfacts.csv


In [2]:
import csv

def detect_delimiter(path, sample_bytes=65536):
    with open(path, "r", encoding="utf-8", errors="replace") as f:
        sample = f.read(sample_bytes)
    try:
        return csv.Sniffer().sniff(sample, delimiters=[",", "\t", ";", "|"]).delimiter
    except Exception:
        return "\t"

DELIM = detect_delimiter(CSV_PATH)
print("Detected delimiter:", repr(DELIM))

TEXT_CANDS = ["ingredients_text","ingredients_text_en","ingredients_text_with_allergens"]
NUTRI_CANDS = ["energy-kcal_100g","energy_100g","proteins_100g","carbohydrates_100g",
               "sugars_100g","fiber_100g","fat_100g","saturated-fat_100g","salt_100g","sodium_100g"]
META_CANDS  = ["code","product_name","brands","categories","compared_to_category","countries_tags"]
LABEL_CANDS = ["nutriscore_grade","nutriscore_score","nova_group","nova_groups_tags",
               "additives_n","additives_tags","allergens","allergens_tags"]


Detected delimiter: '\t'


In [3]:
head = pd.read_csv(
    CSV_PATH, sep=DELIM, nrows=2000, engine="python",
    quoting=csv.QUOTE_NONE, escapechar="\\", on_bad_lines="skip", dtype=str
)

text_col = next((c for c in TEXT_CANDS if c in head.columns), None)
nutri_cols = [c for c in NUTRI_CANDS if c in head.columns]
meta_cols  = [c for c in META_CANDS  if c in head.columns]
label_cols = [c for c in LABEL_CANDS if c in head.columns]

usecols = sorted(set(nutri_cols + meta_cols + label_cols + ([text_col] if text_col else [])))
print("Using columns (first 25):", usecols[:25], "… total:", len(usecols))


Using columns (first 25): ['additives_n', 'additives_tags', 'allergens', 'brands', 'carbohydrates_100g', 'categories', 'code', 'countries_tags', 'energy-kcal_100g', 'energy_100g', 'fat_100g', 'fiber_100g', 'ingredients_text', 'nova_group', 'nutriscore_grade', 'nutriscore_score', 'product_name', 'proteins_100g', 'salt_100g', 'saturated-fat_100g', 'sodium_100g', 'sugars_100g'] … total: 22


In [4]:
def read_chunks(path, sep, usecols, chunksize):
    return pd.read_csv(
        path, sep=sep, usecols=usecols, chunksize=chunksize,
        engine="python", quoting=csv.QUOTE_NONE, escapechar="\\",
        on_bad_lines="skip", dtype=str
    )

TARGET_SAMPLE = 20000
CHUNK_SIZE = 120_000
rows, seen = [], 0

for chunk in read_chunks(CSV_PATH, DELIM, usecols, CHUNK_SIZE):
    for c in nutri_cols:
        if c in chunk.columns:
            chunk[c] = pd.to_numeric(chunk[c], errors="coerce")
    if text_col:
        chunk[text_col] = chunk[text_col].astype(str)

    has_text = chunk[text_col].notna() if text_col else pd.Series(False, index=chunk.index)
    has_nutr = chunk[nutri_cols].count(axis=1) >= 4 if nutri_cols else pd.Series(False, index=chunk.index)
    filt = chunk[has_text | has_nutr]
    if len(filt) == 0:
        continue

    need = TARGET_SAMPLE - seen
    take = min(need, len(filt))
    rows.append(filt.sample(n=take, random_state=42) if take < len(filt) else filt)
    seen += take
    if seen >= TARGET_SAMPLE:
        break

df = pd.concat(rows, ignore_index=True) if rows else pd.DataFrame(columns=usecols)
print(df.shape)
df.head(3)


(20000, 22)


,code,product_name,brands,categories,countries_tags,ingredients_text,allergens,additives_n,additives_tags,nutriscore_score,...,energy-kcal_100g,energy_100g,fat_100g,saturated-fat_100g,carbohydrates_100g,sugars_100g,fiber_100g,proteins_100g,salt_100g,sodium_100g
0,0016300166490,Apple Juice,NaN,NaN,en:france,nan,NaN,NaN,NaN,NaN,...,48.309179,202.0,0.00,0.00,11.835749,10.386473,NaN,0.00,0.0000,0.000
1,0015839009612,"Garlic dill grain free tortilla chips, garlic ...",NaN,Snacks,en:united-states,"Organic cassava flour, high oleic sunflower oi...",NaN,1,en:e270,7,...,500.000000,2092.0,28.57,1.79,57.140000,0.000000,7.1,3.57,0.9825,0.393
2,0012000212000,Nitro,Pepsi,NaN,en:united-states,nan,NaN,NaN,NaN,NaN,...,59.431525,249.0,0.00,0.00,16.279070,16.020672,NaN,0.00,NaN,NaN


In [5]:
import re, numpy as np

def to_tags_list(x):
    if pd.isna(x): return []
    return re.findall(r"en:[a-z0-9\-\+]+", str(x).lower())

df["additives_tags_list"] = df.get("additives_tags", "").apply(to_tags_list)
df["allergens_tags_list"] = df.get("allergens_tags", df.get("allergens","")).apply(to_tags_list)

if "nova_group" not in df.columns and "nova_groups_tags" in df.columns:
    def derive_nova(x):
        L = to_tags_list(x)
        for t in L:
            m = re.search(r"en:([1-4])-", t)
            if m: return int(m.group(1))
        return np.nan
    df["nova_group"] = df["nova_groups_tags"].apply(derive_nova)

if text_col:
    df[text_col] = df[text_col].fillna("").astype(str).str.lower()


In [6]:
ALLERGEN_KEYS = ["en:milk","en:eggs","en:soybeans","en:gluten","en:nuts","en:peanuts","en:sesame"]
for tag in ALLERGEN_KEYS:
    cname = "allergen_" + tag.split(":")[1]
    df[cname] = df["allergens_tags_list"].apply(lambda L: int(tag in L) if isinstance(L, list) else 0)

df["additives_n"] = pd.to_numeric(df.get("additives_n", np.nan), errors="coerce")

def sweetener_flags(row):
    txt = str(row.get(text_col, "")).lower() if text_col else ""
    his = any(k in txt for k in ["aspartame","sucralose","acesulfame","saccharin","neotame","advantame","stevia","reb a","monk fruit","thaumatin","cyclamate"])
    poly = any(k in txt for k in ["erythritol","xylitol","sorbitol","mannitol","maltitol","isomalt","lactitol","hydrogenated starch hydrolysate"])
    for t in row.get("additives_tags_list", []):
        if re.search(r"en:e95[0-9]", t): his = True
        if re.search(r"en:e96[4-8]|en:e420|en:e421|en:e967|en:e968|en:e965", t): poly = True
    return pd.Series({"sweetener_his_flag": int(his), "sweetener_polyol_flag": int(poly), "sweetener_any_flag": int(his or poly)})

df = pd.concat([df, df.apply(sweetener_flags, axis=1)], axis=1)
df.shape


(20000, 34)

In [7]:
num_cols = [c for c in NUTRI_CANDS if c in df.columns]
flag_cols = [c for c in df.columns if c.startswith("allergen_")] + [c for c in ["additives_n","sweetener_any_flag","sweetener_his_flag","sweetener_polyol_flag"] if c in df.columns]
id_cols = [c for c in ["code","product_name","brands","categories","compared_to_category","countries_tags"] if c in df.columns]
text_col = text_col if text_col in df.columns else None

(len(num_cols), len(flag_cols), text_col)


(10, 11, 'ingredients_text')

In [8]:
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.decomposition import TruncatedSVD
from sklearn.feature_extraction.text import TfidfVectorizer

num_pipe = Pipeline([("imp", SimpleImputer(strategy="median")), ("sc", StandardScaler())])
flag_pipe = Pipeline([("imp", SimpleImputer(strategy="most_frequent"))])
txt_pipe  = Pipeline([("tfidf", TfidfVectorizer(min_df=5, max_df=0.98, ngram_range=(1,2),
                                                token_pattern=r"(?u)\b[a-z0-9][a-z0-9\-]+\b",
                                                max_features=10000)),
                      ("svd", TruncatedSVD(n_components=100, random_state=42))])

feature_sets = {}

if len(num_cols) >= 4:
    feature_sets["A_num"] = ColumnTransformer([("num", num_pipe, num_cols)], remainder="drop")

if len(num_cols) >= 4 and len(flag_cols) >= 1:
    feature_sets["B_num_flags"] = ColumnTransformer([("num", num_pipe, num_cols),
                                                     ("flg", flag_pipe, flag_cols)], remainder="drop")

if text_col:
    if len(num_cols) >= 4:
        feature_sets["C_num_text"] = ColumnTransformer([("num", num_pipe, num_cols),
                                                        ("txt", txt_pipe, text_col)], remainder="drop")
    else:
        feature_sets["C_text"] = ColumnTransformer([("txt", txt_pipe, text_col)], remainder="drop")

if text_col and len(flag_cols) >= 1:
    if len(num_cols) >= 4:
        feature_sets["D_num_flags_text"] = ColumnTransformer([("num", num_pipe, num_cols),
                                                              ("flg", flag_pipe, flag_cols),
                                                              ("txt", txt_pipe, text_col)], remainder="drop")
    else:
        feature_sets["D_flags_text"] = ColumnTransformer([("flg", flag_pipe, flag_cols),
                                                          ("txt", txt_pipe, text_col)], remainder="drop")

list(feature_sets.keys())


['A_num', 'B_num_flags', 'C_num_text', 'D_num_flags_text']

In [9]:
from sklearn.cluster import KMeans
from sklearn.mixture import GaussianMixture
from sklearn.metrics import silhouette_score, calinski_harabasz_score, davies_bouldin_score
import numpy as np

def eval_scores(X, labels, cap=4000):
    idx = np.arange(X.shape[0])
    if len(idx) > cap:
        idx = np.random.choice(idx, cap, replace=False)
    return (
        float(silhouette_score(X[idx], labels[idx])),
        float(calinski_harabasz_score(X[idx], labels[idx])),
        float(davies_bouldin_score(X[idx], labels[idx]))
    )

def sweep_kmeans(X, ks):
    out = []
    for k in ks:
        km = KMeans(n_clusters=k, n_init=10, random_state=42)
        lab = km.fit_predict(X)
        sil, ch, db = eval_scores(X, lab)
        out.append(("kmeans", k, sil, ch, db, lab, km))
    return out

def sweep_gmm(X, ks):
    out = []
    for k in ks:
        gm = GaussianMixture(n_components=k, covariance_type="full", random_state=42)
        gm.fit(X)
        lab = gm.predict(X)
        sil, ch, db = eval_scores(X, lab)
        out.append(("gmm", k, sil, ch, db, lab, gm))
    return out


In [10]:
results, embeds = [], {}
k_grid_km = [4,6,8,10]
k_grid_gmm = [4,6,8]

for name, ct in feature_sets.items():
    X = ct.fit_transform(df)
    embeds[name] = X
    for algo,k,sil,ch,db,lab,model in (sweep_kmeans(X, k_grid_km) + sweep_gmm(X, k_grid_gmm)):
        results.append({"feat":name,"algo":algo,"k":k,
                        "silhouette":sil,"calinski_harabasz":ch,"davies_bouldin":db,
                        "labels":lab,"model":model})

res_df = pd.DataFrame([{k:v for k,v in r.items() if k not in ("labels","model")} for r in results])
res_df = res_df.sort_values("silhouette", ascending=False)
res_df.head(10)


,feat,algo,k,silhouette,calinski_harabasz,davies_bouldin
0,A_num,kmeans,4,0.841022,4217.257951,0.712882
1,A_num,kmeans,6,0.809281,2457.947864,0.736280
2,A_num,kmeans,8,0.795598,3525.037468,0.665907
14,C_num_text,kmeans,4,0.785355,1849.296795,0.814658
3,A_num,kmeans,10,0.767677,3115.240201,0.623298
16,C_num_text,kmeans,8,0.739131,1958.427760,0.651216
15,C_num_text,kmeans,6,0.734533,3191.137404,0.662816
17,C_num_text,kmeans,10,0.680441,1019.871005,0.866484
10,B_num_flags,kmeans,10,0.613059,1755.040529,0.896708
9,B_num_flags,kmeans,8,0.603974,2266.785604,0.835451


In [11]:
best = res_df.iloc[0]
best


,0
feat,A_num
algo,kmeans
k,4
silhouette,0.841022
calinski_harabasz,4217.257951
davies_bouldin,0.712882


In [12]:
best_run = None
for r in results:
    if r["feat"]==best["feat"] and r["algo"]==best["algo"] and r["k"]==best["k"]:
        best_run = r; break

X_best = embeds[best["feat"]]
labels_best = best_run["labels"]
model_best = best_run["model"]

df_clusters = df.copy()
df_clusters["_cluster"] = labels_best
df_clusters["_featset"] = best["feat"]
df_clusters["_algo_k"] = f'{best["algo"]}_{best["k"]}'
df_clusters["_cluster"].value_counts().sort_index()


,count
_cluster,
0,19497
1,2
2,6
3,495


In [13]:
def cluster_profile_numeric(data, num_cols, ccol):
    if not num_cols: return pd.DataFrame()
    g = data.groupby(ccol)[num_cols]
    out = pd.concat([
        data.groupby(ccol).size().rename("count"),
        g.median().add_prefix("med_"),
        g.mean().add_prefix("mean_"),
    ], axis=1).reset_index()
    return out

profile = cluster_profile_numeric(df_clusters, num_cols, "_cluster")
profile.head(10)


,_cluster,count,med_energy-kcal_100g,med_energy_100g,med_proteins_100g,med_carbohydrates_100g,med_sugars_100g,med_fiber_100g,med_fat_100g,med_saturated-fat_100g,...,mean_energy-kcal_100g,mean_energy_100g,mean_proteins_100g,mean_carbohydrates_100g,mean_sugars_100g,mean_fiber_100g,mean_fat_100g,mean_saturated-fat_100g,mean_salt_100g,mean_sodium_100g
0,0,19497,264.536849,1105.0,5.19,22.093478,5.000,1.500,5.30,1.670,...,281.544082,1176.575687,8.924758,35.797046,15.685338,3.419665,12.243124,4.524191,1.807389,0.722934
1,1,2,357.000000,1490.0,12.70,69.800000,6.785,3.215,2.77,0.715,...,357.000000,1490.000000,12.700000,69.800000,6.785000,3.215000,2.770000,0.715000,2330.000000,934.500000
2,2,6,26150.000000,106984.0,553.00,2700.000000,100.000,250.000,1050.00,200.000,...,30466.666667,126597.333333,820.666667,3021.000000,1000.600000,2183.833333,1267.716667,210.600000,90.100000,36.060000
3,3,495,1810.000000,7590.0,22.20,131.000000,29.300,6.250,110.50,43.400,...,2232.710039,9385.919028,49.502902,201.031365,109.221450,15.148128,152.106525,53.746298,0.331852,0.132729


In [14]:
top_terms = {}
if text_col:
    from sklearn.feature_extraction.text import TfidfVectorizer
    mask = df_clusters[text_col].notna()
    tf = TfidfVectorizer(min_df=10, max_df=0.99, ngram_range=(1,2),
                         token_pattern=r"(?u)\b[a-z0-9][a-z0-9\-]+\b", max_features=12000)
    Xtf = tf.fit_transform(df_clusters.loc[mask, text_col].astype(str))
    labs = df_clusters.loc[mask, "_cluster"].to_numpy()
    vocab = np.array(tf.get_feature_names_out())
    for c in sorted(df_clusters["_cluster"].unique()):
        idx = np.where(labs == c)[0]
        if len(idx)==0: top_terms[c]=[]; continue
        centroid = Xtf[idx].mean(axis=0).A1
        ix = np.argsort(centroid)[-15:][::-1]
        top_terms[c] = vocab[ix].tolist()

terms_df = pd.DataFrame({"cluster": list(top_terms.keys()), "top_terms": [top_terms[k] for k in top_terms]}) if text_col else pd.DataFrame()
terms_df.head(10)


,cluster,top_terms
0,0,"[nan, salt, acid, water, oil, and, sugar, corn..."
1,1,"[cheese, mix, niacin ferrous, ferrous sulfate,..."
2,2,"[nan, milk, cheese, corn, oil, whey, or sunflo..."
3,3,"[oil, salt, milk, and, sugar, organic, cheese,..."


In [17]:
from sklearn.neighbors import NearestNeighbors
from pathlib import Path
import numpy as np
import pandas as pd
import re

# Build a stable slug for filenames
def slugify(s):
    s = str(s)
    s = s.lower()
    s = re.sub(r"[^a-z0-9\-_]+", "-", s).strip("-")
    return s or "na"

# Nearest neighbors over the best embedding
nn = NearestNeighbors(metric="cosine", n_neighbors=10).fit(X_best)

def similar_by_idx(i, pool, X, nn, same_cluster=True, k=8, dedup_eps=1e-12):
    i = int(max(0, min(i, X.shape[0]-1)))
    q = X[i:i+1]  # 2D
    d, ind = nn.kneighbors(q, n_neighbors=min(k+1, X.shape[0]))
    c0 = int(pool.iloc[i]["_cluster"])
    rows, seen = [], set([i])  # avoid self and exact dup indices
    for j, dist in zip(ind[0], d[0]):
        if j in seen:
            continue
        seen.add(j)
        if same_cluster and int(pool.iloc[j]["_cluster"]) != c0:
            continue
        # optionally skip exact-duplicate neighbors (cosine_dist ~ 0 with identical vectors)
        if dist < dedup_eps:
            continue
        rows.append({
            "row": int(j),
            "cosine_dist": float(dist),
            "product_name": pool.iloc[j].get("product_name",""),
            "brands": pool.iloc[j].get("brands",""),
            "cluster": int(pool.iloc[j]["_cluster"])
        })
        if len(rows) >= k:
            break
    return pd.DataFrame(rows)

# Example neighbor list from the largest cluster
largest_cluster = int(df_clusters["_cluster"].value_counts().idxmax())
cand_idx = df_clusters.index[df_clusters["_cluster"] == largest_cluster].tolist()
i = cand_idx[len(cand_idx)//2] if cand_idx else 0
sim_df = similar_by_idx(i, df_clusters, X_best, nn, same_cluster=True, k=8)
display(sim_df.head(10))

# ---- Save artifacts ----
out_dir = Path("/content/drive/MyDrive/off_unsup")
out_dir.mkdir(exist_ok=True, parents=True)

feat_slug = slugify(best["feat"])
algo_slug = slugify(best["algo"])
k_slug = str(int(best["k"]))

assign_path = out_dir / f"cluster_assignments_{feat_slug}_{algo_slug}{k_slug}.parquet"
summary_path = out_dir / f"cluster_summary_{feat_slug}_{algo_slug}{k_slug}.parquet"
terms_path = out_dir / f"cluster_top_terms_{feat_slug}_{algo_slug}{k_slug}.parquet"

keep_cols = (
    id_cols
    + ([text_col] if text_col else [])
    + num_cols
    + flag_cols
    + ["_cluster","_featset","_algo_k"]
)
keep_cols = [c for c in keep_cols if c in df_clusters.columns]

df_clusters[keep_cols].to_parquet(assign_path, index=False)
if isinstance(profile, pd.DataFrame) and len(profile):
    profile.to_parquet(summary_path, index=False)
if text_col and isinstance(terms_df, pd.DataFrame) and len(terms_df):
    terms_df.to_parquet(terms_path, index=False)

print("Wrote:")
print(str(assign_path))
print(str(summary_path))
print(str(terms_path))


,row,cosine_dist,product_name,brands,cluster
0,15310,0.000014,No pulp 100% premium orange juice,NaN,0
1,16478,0.000014,"Premium juice, orange",NaN,0
2,15789,0.000014,Orange juice,NaN,0
3,16504,0.000020,With Pulp 100% Premium Orange Juice From Conce...,"Florida's Natural, Floridas Natural",0
4,14064,0.000073,100% orange juice from concentrate,NaN,0
5,5346,0.000099,Diced peaches & pears in organic pear juice fr...,NaN,0
6,1572,0.000144,Organic Pears,Earth's Best,0
7,11856,0.000149,"100% orange juice, some pulp",NaN,0


Wrote:
/content/drive/MyDrive/off_unsup/cluster_assignments_a_num_kmeans4.parquet
/content/drive/MyDrive/off_unsup/cluster_summary_a_num_kmeans4.parquet
/content/drive/MyDrive/off_unsup/cluster_top_terms_a_num_kmeans4.parquet


In [19]:
import numpy as np
import pandas as pd
from sklearn.neighbors import NearestNeighbors

def cluster_tags_from_profile(row):
    tags = []
    sugar  = row.get("med_sugars_100g",  row.get("mean_sugars_100g",  np.nan))
    fat    = row.get("med_fat_100g",     row.get("mean_fat_100g",     np.nan))
    satfat = row.get("med_saturated-fat_100g", row.get("mean_saturated-fat_100g", np.nan))
    protein= row.get("med_proteins_100g",row.get("mean_proteins_100g",np.nan))
    sodium = row.get("med_sodium_100g",  row.get("mean_sodium_100g",  np.nan))
    fiber  = row.get("med_fiber_100g",   row.get("mean_fiber_100g",   np.nan))

    if pd.notna(sugar) and sugar >= 15: tags.append("high-sugar")
    elif pd.notna(sugar) and sugar <= 5: tags.append("low-sugar")
    if pd.notna(fat) and fat >= 17: tags.append("high-fat")
    if pd.notna(satfat) and satfat >= 5: tags.append("high-sat-fat")
    if pd.notna(protein) and protein >= 10: tags.append("high-protein")
    if pd.notna(fiber) and fiber >= 5: tags.append("high-fiber")
    if pd.notna(sodium) and sodium >= 600: tags.append("high-sodium")
    elif pd.notna(sodium) and sodium <= 120: tags.append("low-sodium")

    return ", ".join(tags) if tags else "balanced"

def pick_terms(c, terms_df, n=8):
    if terms_df is None or terms_df.empty: return ""
    row = terms_df[terms_df["cluster"] == c]
    if row.empty: return ""
    terms = row["top_terms"].iloc[0]
    if isinstance(terms, (list, tuple)):
        return ", ".join([t for t in terms[:n] if t and t != "nan"])
    return str(terms)[:120]

print("=== Unsupervised Summary ===")
print(f"Best feature set: {best['feat']}")
print(f"Best algorithm / K: {best['algo']} / {int(best['k'])}")
print(f"Silhouette: {best['silhouette']:.3f} | Calinski–Harabasz: {best['calinski_harabasz']:.1f} | Davies–Bouldin: {best['davies_bouldin']:.3f}")
print()

# Build compact cluster table
prof = profile.copy() if isinstance(profile, pd.DataFrame) and not profile.empty else pd.DataFrame()
if not prof.empty:
    prof["_tags"] = prof.apply(cluster_tags_from_profile, axis=1)
else:
    prof = pd.DataFrame({"_cluster": sorted(df_clusters["_cluster"].unique())})
    prof["_tags"] = "n/a"

# Attach counts
if "count" not in prof.columns:
    cnt = df_clusters["_cluster"].value_counts().sort_index().rename("count").reset_index().rename(columns={"index":"_cluster"})
    prof = prof.merge(cnt, on="_cluster", how="left")

# Attach top terms
if 'terms_df' in globals() and isinstance(terms_df, pd.DataFrame) and not terms_df.empty:
    prof["_top_terms"] = prof["_cluster"].apply(lambda c: pick_terms(c, terms_df, n=8))
else:
    prof["_top_terms"] = ""

# Select view
show_cols = ["_cluster", "count", "_tags"]
for k in ["med_sugars_100g","med_proteins_100g","med_fat_100g","med_sodium_100g","med_fiber_100g"]:
    if k in prof.columns: show_cols.append(k)
show_cols.append("_top_terms")

summary_view = prof[show_cols].sort_values("count", ascending=False).reset_index(drop=True)
summary_view.columns = [c.replace("med_","").replace("_100g","/100g") for c in summary_view.columns]

print("Cluster overview:")
display(summary_view.head(12))

# Similar items demo from largest cluster
largest_cluster = int(summary_view.iloc[0]["_cluster"])
cand_idx = df_clusters.index[df_clusters["_cluster"] == largest_cluster].tolist()
if len(cand_idx) >= 1:
    i = int(cand_idx[len(cand_idx)//2])  # representative row
    if 'nn' not in globals():
        nn = NearestNeighbors(metric="cosine", n_neighbors=10).fit(X_best)
    q = X_best[i:i+1]  # 2D slice
    d, ind = nn.kneighbors(q, n_neighbors=min(10, X_best.shape[0]))

    rec = []
    for j, dist in zip(ind[0], d[0]):
        if j == i:
            continue
        if int(df_clusters.iloc[j]["_cluster"]) != largest_cluster:
            continue
        if dist < 1e-12:  # skip exact duplicates
            continue
        rec.append({
            "row": int(j),
            "cosine_dist": float(dist),
            "product_name": df_clusters.iloc[j].get("product_name",""),
            "brands": df_clusters.iloc[j].get("brands",""),
        })
        if len(rec) >= 5:
            break

    rec_df = pd.DataFrame(rec)
    print(f"\nSample similar items within cluster {largest_cluster}:")
    display(rec_df)
else:
    print("\n(no items found in largest cluster)")


=== Unsupervised Summary ===
Best feature set: A_num
Best algorithm / K: kmeans / 4
Silhouette: 0.841 | Calinski–Harabasz: 4217.3 | Davies–Bouldin: 0.713

Cluster overview:


,_cluster,count,_tags,sugars/100g,proteins/100g,fat/100g,sodium/100g,fiber/100g,_top_terms
0,0,19497,"low-sugar, low-sodium",5.000,5.19,5.30,0.21400,1.500,"salt, acid, water, oil, and, sugar, corn"
1,3,495,"high-sugar, high-fat, high-sat-fat, high-prote...",29.300,22.20,110.50,0.00178,6.250,"oil, salt, milk, and, sugar, organic, cheese, ..."
2,2,6,"high-sugar, high-fat, high-sat-fat, high-prote...",100.000,553.00,1050.00,27.00000,250.000,"milk, cheese, corn, oil, whey, or sunflower, c..."
3,1,2,"high-protein, high-sodium",6.785,12.70,2.77,934.50000,3.215,"cheese, mix, niacin ferrous, ferrous sulfate, ..."



Sample similar items within cluster 0:


,row,cosine_dist,product_name,brands
0,15310,0.000014,No pulp 100% premium orange juice,NaN
1,16478,0.000014,"Premium juice, orange",NaN
2,15789,0.000014,Orange juice,NaN
3,16504,0.000020,With Pulp 100% Premium Orange Juice From Conce...,"Florida's Natural, Floridas Natural"
4,14064,0.000073,100% orange juice from concentrate,NaN
